# RoPE — Rotary Position Embedding (Su et al., 2021)RoPE is the positional encoding used by nearly every modern LLM: LLaMA, Qwen, Mistral,Gemma, DeepSeek, and more. It elegantly solves the problems of all previous approaches.This notebook covers:1. The core idea: encoding position as rotation2. Why rotation makes relative position fall out naturally3. Step-by-step implementation (matching your `rope.py`)4. Visualizing what RoPE does to Q and K vectors5. The multi-frequency structure and its connection to sinusoidal PE6. Why RoPE is compatible with KV-cache7. Context length scaling: Position Interpolation, NTK-aware, YaRN

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport matplotlib.pyplot as pltimport numpy as npimport syssys.path.insert(0, '../qwen_n')from rope import apply_rope, compute_rope_params%matplotlib inline

## Part 1: The Core Idea — Position as RotationPrevious approaches **add** position information to embeddings:```input = token_embedding + position_encoding```RoPE instead **rotates** the query and key vectors based on their position:```q_rotated = rotate(q, position * θ)k_rotated = rotate(k, position * θ)```### Why rotation?Remember from the sinusoidal PE notebook: a sin/cos pair at position `pos` is just a pointon a unit circle at angle `pos × ω`. Shifting by offset `k` is a rotation by `k × ω`.RoPE takes this idea and applies it directly to Q and K. Instead of adding a positionvector to the embedding (which mixes position and content), it rotates the Q/K vectorsin 2D subspaces.### The 2D case firstFor a single pair of dimensions, rotating vector `(x₁, x₂)` by angle `θ`:$$R(\theta) \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} x_1 \cos\theta - x_2 \sin\theta \\ x_1 \sin\theta + x_2 \cos\theta \end{pmatrix}$$Let's see this visually:

In [ ]:
# Visualize 2D rotationfig, axes = plt.subplots(1, 3, figsize=(16, 5))# A single 2D vectorq = np.array([1.0, 0.3])# Rotate it by different angles (positions)for ax, title, positions in zip(axes,     ['Positions 0-3', 'Positions 0-7', 'Positions 0-15'],    [range(4), range(8), range(16)]):        theta = 0.5  # frequency for this dimension pair        ax.set_xlim(-1.5, 1.5)    ax.set_ylim(-1.5, 1.5)    ax.set_aspect('equal')    ax.grid(True, alpha=0.3)    ax.axhline(y=0, color='k', linewidth=0.5)    ax.axvline(x=0, color='k', linewidth=0.5)        colors = plt.cm.viridis(np.linspace(0, 1, len(positions)))        for pos, color in zip(positions, colors):        angle = pos * theta        cos_a, sin_a = np.cos(angle), np.sin(angle)        q_rot = np.array([q[0]*cos_a - q[1]*sin_a, q[0]*sin_a + q[1]*cos_a])                ax.arrow(0, 0, q_rot[0]*0.9, q_rot[1]*0.9, head_width=0.05,                 head_length=0.03, fc=color, ec=color, alpha=0.8)        ax.annotate(f'pos={pos}', xy=(q_rot[0]*1.05, q_rot[1]*1.05), fontsize=7, color=color)        ax.set_title(title, fontsize=11)    ax.set_xlabel('Dimension 0')    ax.set_ylabel('Dimension 1')plt.suptitle('RoPE: Same vector rotated by different amounts based on position', fontsize=13, y=1.02)plt.tight_layout()plt.show()print("Each position rotates the vector by a different angle.")print("The CONTENT (vector magnitude and direction) is preserved.")print("Only the ANGLE changes — encoding position without destroying content.")

## Part 2: Why Rotation Gives Relative Position for FreeThis is the key mathematical insight. When we compute the attention score:$$q_m^T k_n = (R(m\theta) \cdot q)^T (R(n\theta) \cdot k)$$Since rotation matrices are orthogonal ($R^T = R^{-1}$):$$= q^T R(m\theta)^T R(n\theta) k = q^T R((n-m)\theta) k$$The attention score depends only on the **relative distance** $(n - m)$, not on theabsolute positions $m$ and $n$ individually!This is the same property that sinusoidal PE had in theory (PE(pos+k) = rotation of PE(pos)),but RoPE achieves it **directly** in the Q·K dot product, without relying on the modelto learn it through W_q and W_k.Let's verify this numerically:

In [ ]:
torch.manual_seed(42)head_dim = 8# Create a query and key vector (content, before rotation)q = torch.randn(head_dim)k = torch.randn(head_dim)def rotate_2d_pairs(x, pos, theta_base=10000):    """Apply RoPE rotation to a single vector at a given position."""    d = x.shape[0]    result = torch.zeros_like(x)    for i in range(d // 2):        theta = pos / (theta_base ** (2 * i / d))        cos_t, sin_t = torch.cos(torch.tensor(theta)), torch.sin(torch.tensor(theta))        result[2*i]   = x[2*i] * cos_t - x[2*i+1] * sin_t        result[2*i+1] = x[2*i] * sin_t + x[2*i+1] * cos_t    return result# Test: q at position m, k at position n# The dot product should depend only on (m - n)pairs_to_test = [    (5, 3),    # distance = 2    (10, 8),   # distance = 2    (100, 98), # distance = 2    (0, 2),    # distance = -2]print("Testing: q_m · k_n should depend only on (m - n)")print("=" * 60)for m, n in pairs_to_test:    q_rot = rotate_2d_pairs(q, m)    k_rot = rotate_2d_pairs(k, n)    score = q_rot @ k_rot    print(f"  m={m:3d}, n={n:3d}, distance={m-n:+d}, score = {score.item():.6f}")print()print("All pairs with distance=2 give the EXACT SAME score!")print("This is relative position encoding without any learned parameters.")print()print("Note: distance=-2 gives a different score — RoPE is direction-aware.")print("This is correct: 'the word 2 positions BEFORE me' ≠ '2 positions AFTER me'.")

## Part 3: Full RoPE Implementation — Step by StepRoPE applies the 2D rotation to **each pair of dimensions** independently, with each pairhaving a different frequency (just like sinusoidal PE).For a head dimension of `d`, we have `d/2` pairs, each rotating at frequency:$$\theta_i = \frac{1}{\text{base}^{2i/d}}$$where `base = 10000` (same as sinusoidal PE).Let's build it from scratch, then verify it matches your `rope.py`:

In [ ]:
def rope_from_scratch(x, position):    """    Apply RoPE to a single vector x at a given position.    This is the explicit, educational version.        x: (head_dim,) tensor    position: integer    """    head_dim = x.shape[0]    base = 10000.0    result = torch.zeros_like(x)        for i in range(head_dim // 2):        # Each pair (2i, 2i+1) rotates at a different frequency        theta_i = 1.0 / (base ** (2 * i / head_dim))        angle = position * theta_i                cos_a = torch.cos(torch.tensor(angle))        sin_a = torch.sin(torch.tensor(angle))                # 2D rotation of the pair (x[2i], x[2i+1])        result[2*i]   = x[2*i] * cos_a - x[2*i+1] * sin_a        result[2*i+1] = x[2*i] * sin_a + x[2*i+1] * cos_a        return result# Now the vectorized version (what your rope.py does)def rope_vectorized(x, cos, sin):    """    Vectorized RoPE — equivalent to rope_from_scratch but fast.        The trick: instead of rotating pairs (x[2i], x[2i+1]),    split into first half and second half: x1 = x[:d/2], x2 = x[d/2:]        Then: rotated = x * cos + rotate_half(x) * sin    where rotate_half(x) = [-x2, x1]        This works because the cos/sin are duplicated: [θ₀,θ₁,...,θ₀,θ₁,...]    """    head_dim = x.shape[-1]    x1 = x[..., :head_dim//2]    x2 = x[..., head_dim//2:]    rotated = torch.cat((-x2, x1), dim=-1)    return x * cos + rotated * sin# Verify they matchtorch.manual_seed(42)head_dim = 16x = torch.randn(head_dim)pos = 7# Method 1: explicit loopresult_explicit = rope_from_scratch(x, pos)# Method 2: vectorized (matching your rope.py)cos, sin = compute_rope_params(head_dim, context_length=128)cos_pos = cos[pos]  # (head_dim,)sin_pos = sin[pos]result_vectorized = rope_vectorized(x, cos_pos, sin_pos)print(f"Explicit result:    {result_explicit[:6].numpy().round(4)}")print(f"Vectorized result:  {result_vectorized[:6].numpy().round(4)}")print(f"Max difference:     {(result_explicit - result_vectorized).abs().max().item():.2e}")print()print("They match! The vectorized version is just a faster way to compute the same rotations.")

### Understanding the vectorized trickYour `rope.py` uses this pattern:```pythonx1 = x[..., :head_dim//2]x2 = x[..., head_dim//2:]rotated = torch.cat((-x2, x1), dim=-1)x_rotated = x * cos + rotated * sin```This looks different from rotating pairs `(x[2i], x[2i+1])`. Here's why it's equivalent:The key is how `compute_rope_params` builds the angles:```pythonangles = torch.cat([angles, angles], dim=1)  # duplicate!```So `cos` and `sin` have the pattern: `[θ₀, θ₁, ..., θ_{d/2-1}, θ₀, θ₁, ..., θ_{d/2-1}]`And the rotation becomes:- First half:  `x1 * cos[:d/2] + (-x2) * sin[:d/2]`  →  `x1·cos(θ) - x2·sin(θ)`- Second half: `x2 * cos[d/2:] + x1 * sin[d/2:]`     →  `x2·cos(θ) + x1·sin(θ)`Which is exactly the 2D rotation `R(θ)·[x1, x2]`! The split is just along the half-dimensionboundary instead of interleaved pairs — mathematically identical.

In [ ]:
# Let's visualize this equivalencehead_dim = 8x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])print("Original x:", x.numpy())print()print("=== Pair-wise view (conceptual) ===")print("  Pair 0: (x[0], x[1]) = (1, 2)  rotated by θ₀")print("  Pair 1: (x[2], x[3]) = (3, 4)  rotated by θ₁")print("  Pair 2: (x[4], x[5]) = (5, 6)  rotated by θ₂")print("  Pair 3: (x[6], x[7]) = (7, 8)  rotated by θ₃")print()print("=== Half-split view (implementation) ===")x1 = x[:4]x2 = x[4:]print(f"  x1 (first half):  {x1.numpy()}  →  these are the 'first' element of each pair")print(f"  x2 (second half): {x2.numpy()}  →  these are the 'second' element of each pair")print()print("  Pair 0: (x1[0], x2[0]) = (1, 5)  rotated by θ₀")print("  Pair 1: (x1[1], x2[1]) = (2, 6)  rotated by θ₁")print("  Pair 2: (x1[2], x2[2]) = (3, 7)  rotated by θ₂")print("  Pair 3: (x1[3], x2[3]) = (4, 8)  rotated by θ₃")print()print("Different pairing, but SAME rotation operation per pair.")print("The half-split is just more GPU-friendly (contiguous memory).")

## Part 4: Visualizing What RoPE Does to Q and KLet's see how RoPE transforms the query and key vectors across positions,and how this affects the attention scores:

In [ ]:
torch.manual_seed(42)head_dim = 64seq_len = 32batch = 1n_heads = 1# Random Q and K (before RoPE)Q = torch.randn(batch, n_heads, seq_len, head_dim)K = torch.randn(batch, n_heads, seq_len, head_dim)# Compute RoPE parameterscos, sin = compute_rope_params(head_dim, context_length=seq_len)# Apply RoPEQ_rope = apply_rope(Q, cos, sin)K_rope = apply_rope(K, cos, sin)# Attention scores before and after RoPEscores_before = (Q @ K.transpose(-2, -1) / (head_dim ** 0.5))[0, 0]scores_after = (Q_rope @ K_rope.transpose(-2, -1) / (head_dim ** 0.5))[0, 0]fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))im0 = axes[0].imshow(scores_before.detach().numpy(), cmap='RdBu_r', aspect='auto')axes[0].set_title('Attention Scores WITHOUT RoPE', fontsize=11)axes[0].set_xlabel('Key position')axes[0].set_ylabel('Query position')plt.colorbar(im0, ax=axes[0], shrink=0.8)im1 = axes[1].imshow(scores_after.detach().numpy(), cmap='RdBu_r', aspect='auto')axes[1].set_title('Attention Scores WITH RoPE', fontsize=11)axes[1].set_xlabel('Key position')axes[1].set_ylabel('Query position')plt.colorbar(im1, ax=axes[1], shrink=0.8)# Show the difference — should reveal the positional structurediff = scores_after - scores_beforeim2 = axes[2].imshow(diff.detach().numpy(), cmap='RdBu_r', aspect='auto')axes[2].set_title('Difference (RoPE contribution)', fontsize=11)axes[2].set_xlabel('Key position')axes[2].set_ylabel('Query position')plt.colorbar(im2, ax=axes[2], shrink=0.8)plt.suptitle('How RoPE Reshapes Attention Scores', fontsize=13, y=1.03)plt.tight_layout()plt.show()print("WITHOUT RoPE: attention is purely content-based (random-looking)")print("WITH RoPE: a diagonal structure emerges — nearby tokens attend more to each other")print("The difference shows the pure positional bias that RoPE introduces")

### The natural distance decayRoPE creates a natural decay in attention scores as distance increases.This happens because distant tokens have their vectors rotated by very differentamounts — the high-frequency dimensions oscillate rapidly and tend to cancel outin the dot product.Let's measure this decay:

In [ ]:
# For a fixed query at position 0, measure attention score to keys at all positionstorch.manual_seed(42)head_dim = 128max_pos = 256q = torch.randn(head_dim)k = torch.randn(head_dim)cos, sin = compute_rope_params(head_dim, context_length=max_pos)# Rotate q at position 0q_at_0 = q * cos[0] + torch.cat((-q[head_dim//2:], q[:head_dim//2])) * sin[0]scores = []for pos in range(max_pos):    k_at_pos = k * cos[pos] + torch.cat((-k[head_dim//2:], k[:head_dim//2])) * sin[pos]    score = (q_at_0 @ k_at_pos).item() / (head_dim ** 0.5)    scores.append(score)fig, axes = plt.subplots(1, 2, figsize=(14, 4))axes[0].plot(scores, color='#3498db', linewidth=1.5, alpha=0.8)axes[0].set_xlabel('Key Position (distance from query at pos 0)')axes[0].set_ylabel('Attention Score')axes[0].set_title('RoPE: Attention Score vs Distance')axes[0].axhline(y=0, color='k', linewidth=0.5, linestyle='--')# Smoothed version to see the envelopewindow = 10smoothed = np.convolve(np.abs(scores), np.ones(window)/window, mode='valid')axes[1].plot(smoothed, color='#e74c3c', linewidth=2)axes[1].set_xlabel('Distance')axes[1].set_ylabel('|Attention Score| (smoothed)')axes[1].set_title('RoPE: Distance Decay Envelope')plt.tight_layout()plt.show()print("The raw scores oscillate (due to the rotation), but the ENVELOPE decays.")print("This means RoPE naturally makes distant tokens less influential —")print("a built-in locality bias without any learned parameters.")

## Part 5: The Multi-Frequency StructureJust like sinusoidal PE, RoPE uses different frequencies for different dimension pairs.This is what gives it the "clock" property — fast-changing dimensions for local position,slow-changing dimensions for global position.$$\theta_i = \frac{1}{10000^{2i/d}}$$- Pair 0 (i=0): $\theta = 1.0$ → completes a full rotation every $2\pi \approx 6.3$ positions- Last pair: $\theta \approx 0.0001$ → barely rotates even over thousands of positions

In [ ]:
head_dim = 128base = 10000# Compute frequencies for each dimension pairfreqs = []wavelengths = []for i in range(head_dim // 2):    theta = 1.0 / (base ** (2 * i / head_dim))    wavelength = 2 * np.pi / theta    freqs.append(theta)    wavelengths.append(wavelength)fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].semilogy(freqs, color='#3498db', linewidth=2)axes[0].set_xlabel('Dimension pair index (i)')axes[0].set_ylabel('Frequency θᵢ (log scale)')axes[0].set_title('RoPE Frequencies per Dimension Pair')axes[0].grid(True, alpha=0.3)axes[1].semilogy(wavelengths, color='#e74c3c', linewidth=2)axes[1].set_xlabel('Dimension pair index (i)')axes[1].set_ylabel('Wavelength (positions per full rotation, log scale)')axes[1].set_title('RoPE Wavelengths per Dimension Pair')axes[1].axhline(y=4096, color='green', linestyle='--', alpha=0.7, label='4K context')axes[1].axhline(y=128000, color='orange', linestyle='--', alpha=0.7, label='128K context')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f"Fastest pair (i=0):  wavelength = {wavelengths[0]:.1f} positions")print(f"Slowest pair (i={head_dim//2-1}): wavelength = {wavelengths[-1]:.0f} positions")print()print("Pairs with wavelength < context_length: encode LOCAL position (many cycles)")print("Pairs with wavelength > context_length: encode GLOBAL position (< 1 cycle)")print()print("For 4K context: ~40 pairs are 'local', ~24 are 'global'")print("This is important for understanding why scaling is needed for longer contexts.")

## Part 6: RoPE and KV-Cache CompatibilityA crucial practical advantage: RoPE is applied to Q and K **independently** at each position.This means when using KV-cache for autoregressive generation:1. At step `t`, you only compute Q for the new token at position `t`2. The cached K values from previous positions are already rotated correctly3. No need to recompute anything — just rotate the new Q and append the new KWith additive PE (sinusoidal or learned), the position is baked into the input embeddingbefore the attention layers. This also works with KV-cache, but RoPE's approach is cleanerbecause position is applied at the Q/K level, not the embedding level.Also: RoPE is applied to Q and K only, **not to V**. This means the value vectors carrypure content without any positional distortion — the position information only affects*which* tokens attend to each other, not *what information* gets passed.

In [ ]:
# Demonstrate KV-cache compatibilitytorch.manual_seed(42)head_dim = 32cos, sin = compute_rope_params(head_dim, context_length=64)# Simulate autoregressive generation# Step 1: process prompt tokens 0-4prompt_K = torch.randn(1, 1, 5, head_dim)prompt_K_rotated = apply_rope(prompt_K, cos, sin)  # rotates each position correctly# Step 2: new token at position 5new_K = torch.randn(1, 1, 1, head_dim)# Only need to rotate position 5cos_5 = cos[5:6].unsqueeze(0).unsqueeze(0)sin_5 = sin[5:6].unsqueeze(0).unsqueeze(0)new_K_rotated = new_K * cos_5 + torch.cat((-new_K[..., head_dim//2:], new_K[..., :head_dim//2]), dim=-1) * sin_5# Verify: this matches what we'd get by processing all 6 tokens at onceall_K = torch.cat([prompt_K, new_K], dim=2)all_K_rotated = apply_rope(all_K, cos, sin)# Compare position 5cached_result = new_K_rotated[0, 0, 0]full_result = all_K_rotated[0, 0, 5]print("KV-Cache compatibility test:")print(f"  Cached (rotate only new token):  {cached_result[:6].numpy().round(4)}")print(f"  Full (rotate all tokens):        {full_result[:6].numpy().round(4)}")print(f"  Max difference: {(cached_result - full_result).abs().max().item():.2e}")print()print("Identical! Each token's rotation depends only on its OWN position.")print("Previously cached K values don't need to be recomputed.")

## Part 7: Context Length ScalingRoPE works great within the training context length. But what happens beyond it?The problem: dimension pairs with high frequencies (small wavelength) wrap around manytimes within the training range. Beyond training length, the model encounters rotationangles it has never seen — not because the angles are invalid, but because the**combination of angles across all dimension pairs** is out-of-distribution.Three solutions emerged in 2023:### 7a. Position InterpolationCompress positions to fit within training range:```position_new = position × (train_len / target_len)```### 7b. NTK-Aware ScalingScale the base frequency to spread interpolation across dimensions:```base_new = base × (scale ^ (d / (d-2)))```### 7c. YaRNPer-dimension strategy: don't interpolate high-freq dims, fully interpolate low-freq dims,blend in between. Plus attention temperature correction.Let's visualize all three:

In [ ]:
head_dim = 64train_len = 4096target_len = 16384  # 4x extensionscale = target_len / train_len  # 4.0positions = torch.arange(target_len, dtype=torch.float)def get_angles(positions, head_dim, base):    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float) / head_dim))    return positions.unsqueeze(1) * inv_freq.unsqueeze(0)# Original RoPE (no scaling)angles_original = get_angles(positions, head_dim, 10000)# Position Interpolation: compress positionspositions_interp = positions * (train_len / target_len)angles_interp = get_angles(positions_interp, head_dim, 10000)# NTK-Aware: scale the basebase_ntk = 10000 * (scale ** (head_dim / (head_dim - 2)))angles_ntk = get_angles(positions, head_dim, base_ntk)fig, axes = plt.subplots(3, 2, figsize=(16, 12))methods = [    ('Original RoPE (no scaling)', angles_original),    ('Position Interpolation', angles_interp),    ('NTK-Aware Scaling', angles_ntk),]for row, (name, angles) in enumerate(methods):    # High frequency dimension (i=0)    ax_hi = axes[row, 0]    ax_hi.plot(angles[:train_len, 0].numpy() % (2*np.pi), '.', markersize=0.5, color='#3498db', alpha=0.5)    ax_hi.plot(np.arange(train_len, target_len),                angles[train_len:, 0].numpy() % (2*np.pi), '.', markersize=0.5, color='#e74c3c', alpha=0.5)    ax_hi.axvline(x=train_len, color='green', linestyle='--', linewidth=1.5, label='Train boundary')    ax_hi.set_title(f'{name} — High freq (i=0)', fontsize=10)    ax_hi.set_ylabel('Angle mod 2π')    ax_hi.legend(fontsize=8)        # Low frequency dimension (last pair)    ax_lo = axes[row, 1]    last_dim = head_dim // 2 - 1    ax_lo.plot(angles[:train_len, last_dim].numpy(), '.', markersize=0.5, color='#3498db', alpha=0.5)    ax_lo.plot(np.arange(train_len, target_len),               angles[train_len:, last_dim].numpy(), '.', markersize=0.5, color='#e74c3c', alpha=0.5)    ax_lo.axvline(x=train_len, color='green', linestyle='--', linewidth=1.5, label='Train boundary')    ax_lo.set_title(f'{name} — Low freq (i={last_dim})', fontsize=10)    ax_lo.set_ylabel('Angle')    ax_lo.legend(fontsize=8)axes[-1, 0].set_xlabel('Position')axes[-1, 1].set_xlabel('Position')plt.suptitle(f'RoPE Scaling Methods: {train_len} → {target_len} (4× extension)', fontsize=14, y=1.02)plt.tight_layout()plt.show()print("Blue = training range, Red = extrapolated range")print()print("Original: red dots go to unseen angles (especially high-freq wraps differently)")print("Position Interpolation: red dots compressed into blue range, but high-freq loses resolution")print(f"NTK-Aware: base changed from 10000 → {base_ntk:.0f}, preserving high-freq resolution")

### 7d. YaRN — The Per-Dimension StrategyYaRN's key insight: different dimension pairs need different treatment.It splits dimensions into three groups based on their wavelength relative to theoriginal training length:1. **High frequency** (wavelength << train_len): Don't interpolate — these are fine2. **Low frequency** (wavelength >> train_len): Fully interpolate — these need stretching3. **Medium frequency**: Blend between no-interpolation and full interpolationPlus a temperature factor to correct the attention entropy shift.

In [ ]:
def yarn_get_scaling_factor(head_dim, base, train_len, target_len, beta_fast=32, beta_slow=1):    """    Compute per-dimension scaling factors for YaRN.        beta_fast: wavelength threshold for 'high frequency' (no interpolation)    beta_slow: wavelength threshold for 'low frequency' (full interpolation)    """    scale = target_len / train_len        factors = []    for i in range(head_dim // 2):        theta = 1.0 / (base ** (2 * i / head_dim))        wavelength = 2 * np.pi / theta                # Ratio of wavelength to training length        r = wavelength / train_len                if r < beta_fast / (2 * np.pi * (scale - 1)):            # High frequency: don't interpolate (factor = 1.0)            factors.append(1.0)        elif r > beta_slow * train_len / (2 * np.pi):            # Low frequency: fully interpolate (factor = 1/scale)            factors.append(1.0 / scale)        else:            # Medium: smooth blend using a ramp function            # Linear interpolation between 1.0 and 1/scale            low = beta_fast / (2 * np.pi * (scale - 1))            high = beta_slow * train_len / (2 * np.pi)            t = (r - low) / (high - low)  # 0 to 1            factor = (1 - t) * 1.0 + t * (1.0 / scale)            factors.append(factor)        return np.array(factors)# Visualize the per-dimension scalingfactors = yarn_get_scaling_factor(head_dim, 10000, train_len, target_len)fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Scaling factors per dimensionaxes[0].plot(factors, 'o-', color='#9b59b6', markersize=3, linewidth=1.5)axes[0].axhline(y=1.0, color='#2ecc71', linestyle='--', alpha=0.7, label='No interpolation')axes[0].axhline(y=1.0/scale, color='#e74c3c', linestyle='--', alpha=0.7, label='Full interpolation')axes[0].set_xlabel('Dimension pair index (i)')axes[0].set_ylabel('Scaling factor')axes[0].set_title('YaRN: Per-Dimension Scaling Factors')axes[0].legend()axes[0].set_ylim(-0.05, 1.15)# Compare effective frequenciesoriginal_freqs = [1.0 / (10000 ** (2*i/head_dim)) for i in range(head_dim//2)]interp_freqs = [f / scale for f in original_freqs]  # Position Interpolationyarn_freqs = [f * factor for f, factor in zip(original_freqs, factors)]axes[1].semilogy(original_freqs, label='Original', color='#3498db', linewidth=2)axes[1].semilogy(interp_freqs, label='Pos. Interpolation (uniform)', color='#e74c3c', linewidth=2, linestyle='--')axes[1].semilogy(yarn_freqs, label='YaRN (per-dimension)', color='#9b59b6', linewidth=2, linestyle='-.')axes[1].set_xlabel('Dimension pair index (i)')axes[1].set_ylabel('Effective frequency (log scale)')axes[1].set_title('Effective Frequencies: Uniform vs YaRN Scaling')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()print("YaRN preserves high-frequency dimensions (left side) while stretching low-frequency ones (right).")print("Position Interpolation uniformly scales everything — losing local position resolution.")print()print("This is why YaRN achieves better quality at extended context lengths:")print("  - Local attention patterns (syntax, adjacent words) stay sharp")print("  - Global attention patterns (document structure) get extended")

## Summary: The Full Positional Encoding Evolution| Method | Year | Type | Key Idea | Scaling ||--------|------|------|----------|---------|| Sinusoidal PE | 2017 | Absolute, additive | Fixed sin/cos at different frequencies | ❌ Model doesn't generalize || Learned PE | 2018 | Absolute, additive | Learnable lookup table | ❌ Hard crash at boundary || Shaw Relative | 2018 | Relative, additive | Learned distance embeddings in attention | ⚠️ Clipped distances || Transformer-XL | 2019 | Relative, additive | 4-term decomposition + recurrence | ⚠️ Complex, sequential || RoPE | 2021 | Relative, multiplicative | Rotate Q/K by position-dependent angle | ✅ With scaling techniques || ALiBi | 2022 | Relative, additive | Linear distance penalty on scores | ✅ Natural extrapolation || Pos. Interpolation | 2023 | RoPE extension | Compress positions into training range | ✅ Simple, needs fine-tuning || NTK-Aware | 2023 | RoPE extension | Scale base frequency | ✅ Preserves local resolution || YaRN | 2023 | RoPE extension | Per-dimension scaling + temperature | ✅ Best quality, minimal fine-tuning |### Why RoPE won:1. **Elegant math**: relative position falls out naturally from rotation2. **No extra parameters**: just a transformation of Q and K3. **KV-cache friendly**: each position's rotation is independent4. **Scalable**: NTK/YaRN extend it to millions of tokens5. **Simple implementation**: ~20 lines of code (see your `rope.py`)The journey from "add sin/cos to embeddings" to "rotate Q/K in subspaces" took 4 years,but the core insight was there from the beginning: **position is rotation**. 🚀